# Step 17.3 — MLX nn.value_and_grad

## 从标量Loss，得到模型可训练参数的梯度树

17.1解释了逐位置交叉熵，17.2把B×T个监督位置展平，并用等权mean得到scalar loss。本节继续连接：

**模型参数 → raw logits → scalar loss → 参数梯度。**

本节按现有roadmap讲Module parameter tree、loss与gradients的function transformation，以及为什么通过返回的梯度树读取结果。我们自带一个很小的Module，不复制完整MiniGPT，不讲Adam或训练循环。

材料审核方已独立核验17.2实际母版、文档与完整教学回复，核心静态PASS，实际修正0/2。这是材料验收，不是学员掌握或运行反馈。17.3本次初始生成实际修正为 **0/2**。

本母版所有代码单元均未执行、无输出。这里只准备发布材料，不读取或下载语料，不执行Notebook、测试、训练、实验、采样或用户work副本，也不代答练习。**本轮止于17.3，不进入17.4。**

## 0. 新 Work Copy：从本节导入、定义和示例开始

沿用项目的 Python 3.12 / `.venv` kernel 与固定依赖 `mlx==0.32.2`。本节只需：

- `mlx.core as mx`：数组、broadcast、reshape、mean；
- `mlx.nn as nn`：Module、cross-entropy与 `nn.value_and_grad`。

新副本自带小Module的类定义、toy chunks、模型实例、scalar loss函数和变换调用。以后使用时，从imports开始按顺序使用本节cells即可；不依赖73副本的kernel变量，不 `%run` 旧课，不依赖语料缓存或sampler。

需要增加的一条连接是：**loss函数内部必须用模型参数产生logits。** 17.2中的手工logits是固定数组，不能把它原样放到模型参数之外，再声称得到了模型参数的梯度。

In [ ]:
import mlx.core as mx
import mlx.nn as nn

# Imports for a future work copy. All cells in this master are unexecuted.


## 1. Scalar Loss 与 Parameter Gradient：问的是谁对谁的变化率？

**Scalar loss（标量损失）** 是17.2最后得到的单个MLX数组，shape为 `()`。模型可能有很多参数，但我们已经把这一批各位置的损失汇总成一个数。

把模型的可训练参数统称为 \(\theta\)（theta），把固定的一批输入与目标记为X、Y。写成：

$$
L(\theta;X,Y).
$$

这个写法强调：本节考察同一批X、Y下，参数改变时L怎样改变。分号只是把关注的参数与当前固定的数据区分开，不是新的Python语法。

**Partial derivative（偏导数）** \(\partial L/\partial\theta_v\) 表示：固定其他参数，只考察第v个参数分量微小变化时，L的局部变化率。正值表示增大该分量会让L局部上升；负值表示会让L局部下降。这是当前位置附近的敏感度，不是任意大改动的保证。

把所有分量的偏导收集起来，就得到 **gradient（梯度）**，记作 \(\nabla_\theta L\)。对于一个shape为 `(3,)` 的参数数组，梯度也有三个对应分量，shape仍为 `(3,)`。

真实MiniGPT有多个参数数组；代码会按参数树保留它们的归属和shape，不需要先把所有参数拼成一个大向量。这里求的是**可训练参数**的梯度，整数token IDs X、Y继续作为数据传入。

## 2. Module Parameter Tree：容器、路径和数组叶子

**Module（模块）** 保存模型参数和计算规则。沿用4.2中的Python概念：

- `class SharedLogitModel(nn.Module)` 定义一个继承Module的类；
- `super().__init__()` 初始化父类的参数管理状态；
- `self` 指当前实例；
- `__call__` 定义调用 `model(X)` 时使用的计算。

**Dictionary（字典）** 通过键名保存值；`head["bias"]` 表示读取head字典里名为bias的值。我们用一个公开属性 `self.head` 保存字典，其中只有一个浮点数组 `bias`。

MLX可以递归收集公开的 `mx.array` 参数，沿子Module、字典和列表继续向内查找。这样的组织称为 **parameter tree（参数树）**：容器描述结构，末端数组称为 **leaf（叶子）**。本例的叶路径是 `head → bias`，它包含1个参数数组、3个可训练标量。

### 本节的最小模型：所有位置共享一组三类别分数

令这个参数向量为：

$$
\theta=[\theta_0,\theta_1,\theta_2]=[0,0,0].
$$

我们让每个位置都使用同一组分数。这样，四个位置会共同影响这三个参数，适合手算和观察参数树。

这个toy Module只读取 `X.shape`，不根据token内容或上下文预测；它是参数与loss接口的示意，不代表MiniGPT架构。零值只为方便这一个手算例，也不是MiniGPT的初始化建议。

### 使用前解释：`mx.broadcast_to`

**Broadcasting（广播）** 将兼容的数组形状扩展到目标shape。这里 `mx.broadcast_to(bias, (B,T,V))` 把 `bias:(V,)` 用在每个位置，返回 `(B,T,V)`，满足 `logits[b,t,v]=bias[v]`。

它表达的是同一个参数在各位置被共享使用，没有因此产生B×T组独立参数。`B,T=X.shape` 按顺序取输入的两条轴长度；`bias.shape[0]` 是这个向量的长度V。

[MLX broadcast_to API](https://ml-explore.github.io/mlx/build/html/python/_autosummary/mlx.core.broadcast_to.html) · [固定v0.32.2 Module源码](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/nn/layers/base.py)

In [ ]:
class SharedLogitModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.head = {
            "bias": mx.array([0.0, 0.0, 0.0], dtype=mx.float32),
        }

    def __call__(self, X):
        B, T = X.shape
        V = self.head["bias"].shape[0]
        return mx.broadcast_to(self.head["bias"], (B, T, V))


# Definition only. No model or loss has been evaluated in this master.
# This toy uses X.shape only; it does not model token content or context.


### 自带数据：保留17.2的四个监督位置

继续使用两行toy chunks：`[[0,2,1],[1,0,2]]`。`mx.int32` 保存整数token IDs；和前节一样，`[:, :-1]` 取输入，`[:, 1:]` 取已右移目标。

因此X为 `[[0,2],[1,0]]`，Y为 `[[2,1],[0,2]]`，B=2、T=2、N=B×T=4，按 `r=b*T+t` 展平后的Y为 `[2,1,0,2]`。

`SharedLogitModel()` 构造这个类的一个实例，赋给变量model。数据作为普通输入放在模型外；它们不会因为本节需要自包含，就变成模型的公开参数属性。没有随机初始化或随机采样。

In [ ]:
# USER-RUN TOY DATA AND INSTANCE — prepared only, NOT executed.

toy_chunks = mx.array([[0, 2, 1], [1, 0, 2]], dtype=mx.int32)
X = toy_chunks[:, :-1]  # (2, 2)
Y = toy_chunks[:, 1:]   # (2, 2), already shifted

model = SharedLogitModel()

# Parameters are explicit zeros; no RNG, corpus, sampler, or full MiniGPT.


### `parameters()` 与 `trainable_parameters()`

- `model.parameters()` 返回模型收集到的参数树，包括已被冻结的参数。
- `model.trainable_parameters()` 返回其中未被冻结的参数树，供本节的求导变换使用。

**Frozen parameter（冻结参数）** 仍可参与模型计算，但不在这一可训练参数集合中。本例不冻结任何参数，也不添加冻结实验，两棵树都只有同一个bias叶子。

结构示意为：

```python
{"head": {"bias": ...}}  # ... 表示一个shape为(3,)的MLX数组
```

`params["head"]["bias"]` 先进入head字典，再取bias数组；`.shape` 查看该叶子的形状。这里的参数树结构，与17.2在一个张量内合并B、T轴的reshape是两件事。

In [ ]:
# USER-RUN PARAMETER STRUCTURE — prepared only, NOT executed.

params = model.parameters()
trainable = model.trainable_parameters()

print("parameter leaf shape:", params["head"]["bias"].shape)
print("trainable leaf shape:", trainable["head"]["bias"].shape)

# Both leaf shapes are (3,) by construction; these are not runtime results.


## 3. 把参数到Scalar Loss的计算写成一个函数

**Function（函数）** 把输入到结果的计算命名。下面的 `loss_fn(model, X, Y)` 接收模型和数据，`return` 返回这个函数的结果。

关键是把 `logits = model(X)` 放在函数内部，让logits依赖本次函数调用使用的模型参数，然后接上17.2原有的步骤：

| 环节 | 本例shape |
|---|---|
| 参数叶子 `model.head["bias"]` | `(3,)` |
| `model(X)` 的raw logits | `(2,2,3)` |
| 展平logits / targets | `(4,3)` / `(4,)` |
| `cross_entropy(..., reduction="none")` | `(4,)` |
| `mx.mean` | `()` |

继续使用raw logits、整数targets、类别轴 `axis=-1`，不再次shift Y。`mx.reshape` 的两侧保持同一位置顺序；`mx.mean` 汇总全部有效位置，本例均等权。

函数返回MLX的scalar array，让自动微分保留这条数组计算关系；不要在 `return` 时用 `.item()` 或 `float(...)` 把它改成Python数值。

In [ ]:
def loss_fn(model, X, Y):
    logits = model(X)  # Must depend on this model's parameters.
    B, T, V = logits.shape
    N = B * T

    flat_logits = mx.reshape(logits, (N, V))
    flat_targets = mx.reshape(Y, (N,))
    token_losses = nn.losses.cross_entropy(
        flat_logits,
        flat_targets,
        axis=-1,
        reduction="none",
    )
    return mx.mean(token_losses)  # Scalar MLX array, shape ().


# Function definition only. No call is executed in this material-preparation turn.


## 4. 先用这个共享参数小例手算Loss和Gradient

### Loss：同一组参数在四个位置使用

当前 \(\theta=[0,0,0]\)。每个位置的raw logits都是这三个0，因此每个类别的模型概率都是：

$$
p_v=\frac{e^{\theta_v}}{\sum_{u=0}^{2}e^{\theta_u}}=\frac13.
$$

每个位置的正确类别概率也都是1/3，所以四项位置loss均为 \(\ln3\)，平均后：

$$
L=\frac{\ln3+\ln3+\ln3+\ln3}{4}=\ln3.
$$

这里保留了17.2的targets，但改变了logits的来源：现在由参数产生统一的初始分数，因此不沿用前节手工logits的loss数值。这些值仍是数学推导，没有调用模型或运行Notebook。

### Gradient：为什么形状跟参数，而不是跟四个位置？

r表示展平后的监督位置，v表示类别。令 \(q_{r,v}\) 为该位置target的one-hot权重：当 \(Y_{\text{flat}}[r]=v\) 时为1，否则为0。它只是数学记号，代码仍传整数Y。

17.1已给出每个位置的交叉熵：

$$
\ell_r=\ln\left(\sum_{u=0}^{2}e^{\theta_u}\right)-\theta_{y_r}.
$$

这里 \(y_r\) 是第r个位置的正确类别。为看清导数，只需复习两条规则：

- 指数函数的导数为自身：\(\frac{d}{du}e^u=e^u\)；
- 对s>0，自然对数的导数为 \(\frac{d}{ds}\ln s=1/s\)。

再用5.3中的 **chain rule（链式法则）**：复合计算的局部导数相乘。把求和整体暂记为S；固定其余参数，对 \(\theta_v\) 求偏导时，log部分给出 \((1/S)e^{\theta_v}=p_v\)。

后面的 \(-\theta_{y_r}\) 只有在v就是正确类别时才贡献−1，其他情况下贡献0。因此：

$$
\frac{\partial\ell_r}{\partial\theta_v}=p_v-q_{r,v}.
$$

四个位置共享同一个 \(\theta_v\)，而L是四项loss的平均，所以参数梯度也把四处贡献取平均：

$$
\frac{\partial L}{\partial\theta_v}
=\frac14\sum_{r=0}^{3}(p_v-q_{r,v}).
$$

这里 \(p_v\) 不带r下标，是因为这个toy在所有位置共享同一组分数。Targets为 `[2,1,0,2]`，三个类别在四个位置中的出现比例分别为1/4、1/4、1/2：

| 参数分量 | 模型概率 | 对应类别的目标比例 | 参数偏导 |
|---|---|---|---|
| \(\theta_0\) | \(1/3\) | \(1/4\) | \(1/12\) |
| \(\theta_1\) | \(1/3\) | \(1/4\) | \(1/12\) |
| \(\theta_2\) | \(1/3\) | \(1/2\) | \(-1/6\) |

因此：

$$
\boxed{\nabla_\theta L=[1/12,\ 1/12,\ -1/6],\qquad \text{shape }(3,).}
$$

这是一个参数数组的三个分量的偏导。四个监督位置共同贡献，不会让这个参数叶子的梯度变成 `(B,T,V)`。例如最后的负号表示：在当前点附近，其他参数固定时，增大 \(\theta_2\) 会让L下降；本节只解释这一局部敏感度，不实施参数更新。

## 5. nn.value_and_grad：从一个函数得到另一个函数

**Callable（可调用对象）** 是可以用括号和实参调用的对象；这里的 `loss_fn` 是一个函数对象。

回顾Python里的区别：

- `loss_fn`：函数本身，可作为参数传给另一个函数；
- `loss_fn(model, X, Y)`：调用函数，取得它返回的loss。

**Function transformation（函数变换）** 接收函数并产生一个新函数。与5.3中 `mx.value_and_grad` 的思路相同，这次我们希望同时取得value和gradient；新增的是让Module提供它的可训练参数树。

固定MLX0.32.2的接口为 `nn.value_and_grad(model, fn)`：

| 这一层调用中的参数 | 作用 |
|---|---|
| `model` | 指定对哪个Module的可训练参数求导 |
| `loss_fn` | 给出从模型与数据返回scalar loss的函数 |
| 返回值 | 一个新的函数，之后调用它可取得loss和参数梯度树 |

因此先创建变换后的函数。这里传的是 `loss_fn` 本身，不提前把 `loss_fn(...)` 的数值结果传进去。

[官方nn.value_and_grad API](https://ml-explore.github.io/mlx/build/html/python/_autosummary/mlx.nn.value_and_grad.html) · [固定v0.32.2实现](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/nn/utils.py)

In [ ]:
# USER-RUN FUNCTION TRANSFORMATION — prepared only, NOT executed.

loss_and_grad_fn = nn.value_and_grad(model, loss_fn)

# This constructs a callable; it is not a training step or a call to loss_fn.


### 调用新函数：两处model分别做什么？

我们的loss函数明确写作 `loss_fn(model, X, Y)`。MLX的包装器会把新函数收到的普通实参原样交给loss函数，**不会自动替它插入model实参**。因此，之后调用时写：

```python
loss_value, grads = loss_and_grad_fn(model, X, Y)
```

第一次在 `nn.value_and_grad(model, loss_fn)` 中传model，是选择求导参数所属的Module；第二次传model、X、Y，是满足我们自己定义的loss函数签名。这里使用同一个模型实例。

`loss_value, grads = ...` 是Python的 **tuple unpacking（元组解包）**，按顺序接收两个返回项：

- `loss_value`：本次调用的scalar loss，shape `()`；
- `grads`：loss对该模型可训练参数的梯度树。

源码在每次调用时取得 `model.trainable_parameters()`；它不是构造新函数时保存一份固定参数值快照。

这个nn接口是针对Module的包装；前置的 `mx.value_and_grad` 是针对显式传入的数组或数组树的通用变换，本节统一使用nn版本，不混用两者的调用签名。

版本实现内部还会用 `model.update(params)` 把本次求导使用的参数接入模型计算。这里的update是按路径安装参数，不是根据梯度执行学习更新；因此不能把包装器说成“对模型对象完全没有写入”。本节没有学习率、优化器或参数调整步骤。

In [ ]:
# USER-RUN VALUE AND GRADIENT — prepared only, NOT executed in the master.

loss_value, grads = loss_and_grad_fn(model, X, Y)

print("loss shape:", loss_value.shape)
print("gradient leaf shape:", grads["head"]["bias"].shape)
print("loss:", loss_value)
print("bias gradient:", grads["head"]["bias"])

# Hand-derived expectations for the explicit zero parameters:
# loss_value = ln(3), shape ()
# grads["head"]["bias"] = [1/12, 1/12, -1/6], shape (3,)
# These are not observed outputs. No optimizer or parameter-learning update.


### 返回的梯度树：同一路径，对应不同含义的数组

本例可训练参数树与梯度树都具有 `{"head": {"bias": ...}}` 的结构：

| 读取路径 | 含义 | Shape |
|---|---|---|
| `model.trainable_parameters()["head"]["bias"]` | 模型的三个可训练分数参数 | `(3,)` |
| `grads["head"]["bias"]` | L对这三个参数各自的偏导数 | `(3,)` |

在更大的Module中，梯度仍按可训练参数的容器、键路径和叶子shape组织。全参数树可能还包括被冻结的参数，因此不要要求grads包含所有 `parameters()` 路径并把冻结项填零；本例没有冻结项。

本课代码从 **返回值grads** 读取梯度，不依赖给每个参数数组自动附着一个 `.grad` 属性，也不引入 `backward()` 或 `zero_grad()` 的工作方式。梯度数组描述的是导数，本身不是下一组参数值。

Auto-differentiation（自动微分）沿数组运算的求导规则连接参数与loss；这里没有逐参数扰动的有限差分实验，也没有把上述手算当作已执行的数值验证。

## 本节的Shape与工作副本顺序

| 对象 | Shape / 结构 | 作用 |
|---|---|---|
| X、Y | `(2,2)`、`(2,2)` | 普通整数数据；Y已经右移 |
| 参数叶子bias | `(3,)` | 3个共享的可训练标量 |
| logits | `(2,2,3)` | 四个位置共享这组raw scores |
| flat logits / targets | `(4,3)` / `(4,)` | 17.2的监督组织 |
| token losses | `(4,)` | 每个位置的交叉熵 |
| loss value | `()` | 全部位置的等权平均 |
| grads | 嵌套字典，bias叶子 `(3,)` | 对应可训练参数的梯度 |

之后需要使用work副本时：

1. 在仓库根目录更新，用时间戳新文件名与 `cp -n` 复制74母版到已有的 `notebooks/work/`，只打开新副本。
2. 选择项目Python3.12 / `.venv` kernel，从本节imports开始；无需旧kernel或外部数据。
3. 阅读参数与梯度概念、Module/字典/broadcast说明，再使用类定义和自带数据/实例代码。
4. 读参数树API，再使用参数路径的shape观察代码；读scalar loss的函数说明，再定义loss_fn。
5. 阅读手算与函数变换说明，再构造loss_and_grad_fn；理解调用签名与返回解包后，使用最后的调用示例。

这些是未来使用副本的步骤，本轮没有执行。材料准备、手算预期、静态审阅与发布均不表示学员掌握或运行通过。

**本轮只准备并发布17.3，完成即停，不进入17.4。** 不准备Adam、优化器状态、参数更新或训练循环，不返工已通过课节。

累计实际修正：16.1=**1/2**、16.2=**1/2**，16.3至16.5及17.1/17.2各**0/2**，17.3初始生成=**0/2**。16.2的一次包含已记录的超时后完整回复补齐。每节最多两次实际修正，跨轮与换对话不重置；后续只按请求逐小节至大Step20全部完成后停止。

## 参考与课程衔接

- [17.2 Flatten B×T母版](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/c13c768dce7200ff6eefd1fca85384f9ebcc8154/notebooks/73_step17_2_flatten_bt_supervision.ipynb)：逐位置loss与scalar mean。
- [4.2 Module与参数树](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/c13c768dce7200ff6eefd1fca85384f9ebcc8154/notebooks/09_step04_2_parameters_bias_module.ipynb)、[5.3 Gradient与自动微分](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/c13c768dce7200ff6eefd1fca85384f9ebcc8154/notebooks/14_step05_3_gradient_autodiff.ipynb)：本节必要前置回顾。
- [MLX nn.value_and_grad](https://ml-explore.github.io/mlx/build/html/python/_autosummary/mlx.nn.value_and_grad.html)、[Module](https://ml-explore.github.io/mlx/build/html/python/nn/module.html)、[broadcast_to](https://ml-explore.github.io/mlx/build/html/python/_autosummary/mlx.core.broadcast_to.html)：API说明。
- [MLX v0.32.2 nn/utils.py](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/nn/utils.py)、[Module base.py](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/nn/layers/base.py)、[transforms.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/transforms.cpp)、[losses.py](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/nn/losses.py)：固定版本的函数包装、参数收集、梯度树与loss契约依据。在线文档可能更新，项目仍固定MLX0.32.2。
- [Step17课程文档](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/main/docs/step-17-language-model-loss.md) 与 [course roadmap](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/main/docs/course-roadmap.md)。